## Notebook — Camada Gold 

In [0]:
catalog = "cinedata"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 1. dim_movies
- Objetivo: Armazenar os metadados principais e descritivos de cada filme.
- Colunas: sk_movie_id BIGINT (PK), id_filme STRING (Chave natural de origem), titulo STRING, data_lancamento
DATE, ano_lancamento INT, duracao_minutos INT, idioma_original STRING, status_filme STRING, sinopse STRING.


In [0]:
df_info_silver = spark.table(f"{silver_schema}.tb_info_filmes")

df_dim_movies = (
    df_info_silver
    .withColumn("id_filme", F.col("id_filme").cast("string"))
    .filter(F.col("id_filme").isNotNull())
    .dropDuplicates(["id_filme"])
    .withColumn("sk_movie_id", F.row_number().over(Window.orderBy("id_filme")).cast("bigint"))
    .withColumn("data_lancamento", F.col("data_lancamento").cast("date"))
    .withColumn("ano_lancamento", F.col("ano_lancamento").cast("int"))
    .withColumn("duracao_minutos", F.col("duracao_minutos").cast("int"))
    .select("sk_movie_id", "id_filme", "titulo", "data_lancamento", "ano_lancamento",
            "duracao_minutos", "idioma_original", "status_filme", "sinopse")
)

df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_movies")

print(f"dim_movies: {df_dim_movies.count()} filmes")
display(df_dim_movies.limit(10))

## 2. dim_genres
- Objetivo: Catálogo único e deduplicado de todos os gêneros cinematográficos.
- Colunas: sk_genre_id BIGINT (PK), nome_genero STRING.


In [0]:
df_gen_silver = spark.table(f"{silver_schema}.tb_generos")

df_dim_genres = (
    df_gen_silver
    .select(F.col("genero").alias("nome_genero"))
    .filter(F.col("nome_genero").isNotNull())
    .dropDuplicates(["nome_genero"])
    .withColumn("sk_genre_id", F.row_number().over(Window.orderBy("nome_genero")).cast("bigint"))
    .select("sk_genre_id", "nome_genero")
)

df_dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_genres")

print(f"dim_genres: {df_dim_genres.count()} gêneros")
display(df_dim_genres)

## 3. dim_people
- Objetivo: Consolidar todas as pessoas físicas envolvidas na obra.
- Colunas: sk_person_id BIGINT (PK), nome_pessoa STRING, tipo_pessoa STRING (Apenas valores como: 'Ator', 'Diretor', 'Roteirista').


In [0]:
df_pe_silver = spark.table(f"{silver_schema}.tb_pessoas_empresas")

df_dim_people = (
    df_pe_silver
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
    .dropDuplicates(["nome_pessoa", "tipo_pessoa"])
    .withColumn("sk_person_id", F.row_number().over(Window.orderBy("nome_pessoa", "tipo_pessoa")).cast("bigint"))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

df_dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_people")

print(f"dim_people: {df_dim_people.count()} pessoas")
display(df_dim_people.groupBy("tipo_pessoa").count())

## 4. dim_companies
- Objetivo: Catálogo único de produtoras/estúdios.
- Colunas: sk_company_id BIGINT (PK), nome_produtora STRING.


In [0]:
df_dim_companies = (
    df_pe_silver
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .dropDuplicates(["nome_produtora"])
    .withColumn("sk_company_id", F.row_number().over(Window.orderBy("nome_produtora")).cast("bigint"))
    .select("sk_company_id", "nome_produtora")
)

df_dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_companies")

print(f"dim_companies: {df_dim_companies.count()} produtoras")
display(df_dim_companies.limit(10))

## 5. dim_reviews
- Objetivo: Consolidar as avaliações individuais geradas pelos usuários, transformando-as em uma métrica resumida por
filme.
- Colunas:sk_review_id BIGINT (PK), sk_movie_id BIGINT (FK para o filme correspondente), qtd_avaliacoes_usuarios
INT (Contagem), nota_media_usuarios DOUBLE (Média arredondada em 2 casas decimais).


In [0]:
df_rev_silver = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")

df_dim_movies_lk = spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "id_filme")

df_dim_reviews = (
    df_rev_silver
    .withColumn("id_filme", F.col("id_filme").cast("string"))
    .filter(F.col("nota_usuario").isNotNull())
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios"),
    )
    .join(df_dim_movies_lk, on="id_filme", how="inner")
    .withColumn("sk_review_id", F.row_number().over(Window.orderBy("sk_movie_id")).cast("bigint"))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)

df_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_reviews")

print(f"dim_reviews: {df_dim_reviews.count()} filmes com avaliações")
display(df_dim_reviews.limit(10))

## 6. fact_movies_performance
- Grão: Um registro único por filme.
- Objetivo: Centralizar todas as métricas financeiras e de engajamento do filme.
- Chave Estrangeira (FK): sk_movie_id BIGINT (Apontando para a dim_movies).
- Métricas Financeiras (DECIMAL): orcamento_usd DECIMAL(18,2), receita_usd DECIMAL(18,2), lucro_usd
DECIMAL(18,2), orcamento_brl DECIMAL(18,2), receita_brl DECIMAL(18,2), lucro_brl DECIMAL(18,2).
- Métricas de Engajamento (DOUBLE/INT): popularidade DOUBLE, nota_media_tmdb DOUBLE, qtd_votos_tmdb
INT, nota_media_imdb DOUBLE, qtd_votos_imdb INT.
- A tabela fato deve consolidar métricas de filmes lançados sem duplicar grão através dos joins.


In [0]:
df_fin = (
    spark.table(f"{silver_schema}.tb_financeiro_filmes")
    .withColumn("id_filme", F.col("id_filme").cast("string"))
    .dropDuplicates(["id_filme"])
)

df_met = (
    spark.table(f"{silver_schema}.tb_metricas_engajamento")
    .withColumn("id_filme", F.col("id_filme").cast("string"))
    .dropDuplicates(["id_filme"])
)

df_fact = (
    spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "id_filme")
    .join(df_fin, on="id_filme", how="left")
    .join(df_met, on="id_filme", how="left")
    .select(
        "sk_movie_id",
        F.col("orcamento_usd").cast("decimal(18,2)").alias("orcamento_usd"),
        F.col("receita_usd").cast("decimal(18,2)").alias("receita_usd"),
        F.col("lucro_usd").cast("decimal(18,2)").alias("lucro_usd"),
        F.col("orcamento_brl").cast("decimal(18,2)").alias("orcamento_brl"),
        F.col("receita_brl").cast("decimal(18,2)").alias("receita_brl"),
        F.col("lucro_brl").cast("decimal(18,2)").alias("lucro_brl"),
        F.col("popularidade").cast("double").alias("popularidade"),
        F.col("nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
        F.col("qtd_votos_tmdb").cast("int").alias("qtd_votos_tmdb"),
        F.col("nota_media_imdb").cast("double").alias("nota_media_imdb"),
        F.col("qtd_votos_imdb").cast("int").alias("qtd_votos_imdb"),
    )
)

df_fact.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fact_movies_performance")

print(f"fact_movies_performance: {df_fact.count()} filmes")
display(df_fact.limit(10))

## 7. Tabelas-ponte

In [0]:
df_movie_lk = spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "id_filme")

# bridge_movie_genre
df_bridge_genre = (
    spark.table(f"{silver_schema}.tb_generos")
    .withColumn("id_filme", F.col("id_filme").cast("string"))
    .join(spark.table(f"{gold_schema}.dim_genres"), F.col("genero") == F.col("nome_genero"), how="inner")
    .join(df_movie_lk, on="id_filme", how="inner")
    .select("sk_movie_id", "sk_genre_id")
    .dropDuplicates()
)
df_bridge_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_genre")

# bridge_movie_person
df_bridge_person = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .withColumn("id_filme", F.col("id_filme").cast("string"))
    .join(
        spark.table(f"{gold_schema}.dim_people"),
        (F.col("nome_entidade") == F.col("nome_pessoa")) & (F.col("tipo_entidade") == F.col("tipo_pessoa")),
        how="inner",
    )
    .join(df_movie_lk, on="id_filme", how="inner")
    .select("sk_movie_id", "sk_person_id")
    .dropDuplicates()
)
df_bridge_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_person")

# bridge_movie_company
df_bridge_company = (
    spark.table(f"{silver_schema}.tb_pessoas_empresas")
    .filter(F.col("tipo_entidade") == "Produtora")
    .withColumn("id_filme", F.col("id_filme").cast("string"))
    .join(spark.table(f"{gold_schema}.dim_companies"), F.col("nome_entidade") == F.col("nome_produtora"), how="inner")
    .join(df_movie_lk, on="id_filme", how="inner")
    .select("sk_movie_id", "sk_company_id")
    .dropDuplicates()
)
df_bridge_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_company")

print("bridge_movie_genre:", df_bridge_genre.count())
print("bridge_movie_person:", df_bridge_person.count())
print("bridge_movie_company:", df_bridge_company.count())

## 8. Conferência do Star Schema
- `fact_movies_performance` deve ter o mesmo nº de linhas que `dim_movies` e `sk_movie_id` sem repetição.
- Nenhuma chave das pontes pode ficar sem correspondência nas dimensões (órfãs = 0).

In [0]:
fact = spark.table(f"{gold_schema}.fact_movies_performance")
dmov = spark.table(f"{gold_schema}.dim_movies")

display(spark.createDataFrame([
    ("dim_movies", dmov.count(), dmov.select("sk_movie_id").distinct().count()),
    ("fact_movies_performance", fact.count(), fact.select("sk_movie_id").distinct().count()),
], ["tabela", "linhas", "sks_distintas"]))

# órfãs: chaves das pontes que não existem na dimensão correspondente
orf_g = spark.table(f"{gold_schema}.bridge_movie_genre").join(spark.table(f"{gold_schema}.dim_genres"), "sk_genre_id", "left_anti").count()
orf_p = spark.table(f"{gold_schema}.bridge_movie_person").join(spark.table(f"{gold_schema}.dim_people"), "sk_person_id", "left_anti").count()
orf_c = spark.table(f"{gold_schema}.bridge_movie_company").join(spark.table(f"{gold_schema}.dim_companies"), "sk_company_id", "left_anti").count()
print("órfãs genre/person/company:", orf_g, orf_p, orf_c)

spark.sql(f"SHOW TABLES IN {gold_schema}").show(truncate=False)